# EMG2QWERTY Training Notebook

This notebook provides a complete setup for training EMG-to-text models on **Google Colab or locally** (including Mac M1/M2/M3).

**If using Google Colab:**
1. Go to **Runtime → Change runtime type**
2. Select **GPU** as the hardware accelerator
3. Click **Save**

**If using locally:**
- **Linux/Windows with NVIDIA GPU**: CUDA will be used automatically
- **Mac M1/M2/M3**: MPS (Metal Performance Shaders) will be used automatically
- **CPU only**: Training will be slower but still works
- Run this notebook in Jupyter from the repo root directory

The notebook automatically detects the environment and hardware accelerator.

---

## 1. Setup & Configuration

Detect environment and configure paths accordingly.

**Understanding paths:**
- **Google Colab**: `/content/drive/MyDrive/` = Your Google Drive root (same as drive.google.com)
- **Local**: `./logs/` = Logs folder in the repository directory

In [1]:
# Detect environment
try:
    import google.colab
    IN_COLAB = True
    print("🌐 Running in Google Colab")
except ImportError:
    IN_COLAB = False
    print("💻 Running locally")

# Mount Google Drive if in Colab
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    print("✓ Google Drive mounted")
else:
    print("✓ Using local filesystem")

💻 Running locally
✓ Using local filesystem


In [2]:
# Configuration - Adapts automatically to environment
import os

# GitHub repository URL (used only in Colab)
REPO_URL = "https://github.com/sttawm/emg2qwerty.git"

# Dataset download URL (Box shared link)
DATASET_URL = "https://ucla.app.box.com/s/3xc4nwpfjfpo6ydjs94t0v2kuq37d5eg"

if IN_COLAB:
    # === COLAB CONFIGURATION ===
    # EDIT THESE PATHS FOR YOUR TEAM:
    
    # Shared Google Drive folder for logs and checkpoints
    # This folder will appear in your Google Drive and can be shared with team
    SHARED_LOGS_PATH = "/content/drive/MyDrive/dev/emg2qwerty/logs"
    
    # (Optional) Path to dataset in Google Drive if pre-uploaded
    # Set to None to download from Box automatically
    DATASET_PATH = "/content/drive/MyDrive/dev/emg2qwerty/data"
    
    # Working directory (where repo will be cloned)
    WORK_DIR = "/content/emg2qwerty"
    
else:
    # === LOCAL CONFIGURATION ===
    # EDIT THESE PATHS IF NEEDED:
    
    # Logs directory (relative to repo root)
    SHARED_LOGS_PATH = "./logs"
    
    # (Optional) Path to dataset if not in ./data
    # Set to None to check ./data or prompt for setup
    DATASET_PATH = None
    
    # Working directory (current directory, assuming you're in repo root)
    WORK_DIR = "."

print("✓ Configuration set")
print(f"  Environment: {'Google Colab' if IN_COLAB else 'Local'}")
print(f"  Logs will be saved to: {SHARED_LOGS_PATH}")
print(f"  Dataset source: {'Google Drive' if DATASET_PATH else 'Box/Local data folder'}")

✓ Configuration set
  Environment: Local
  Logs will be saved to: ./logs
  Dataset source: Box/Local data folder


---
## 2. Clone Repository

Clone the latest version from GitHub (Colab only - skipped when running locally).

In [3]:
import os

if IN_COLAB:
    # Remove existing clone if present
    if os.path.exists(WORK_DIR):
        print("Removing existing repository...")
        !rm -rf {WORK_DIR}
    
    # Clone repository
    print(f"Cloning repository from {REPO_URL}...")
    !git clone {REPO_URL} {WORK_DIR}
    
    # Change to repo directory
    %cd {WORK_DIR}
    
    print("\n✓ Repository cloned successfully")
    !git log -1 --oneline
else:
    print("✓ Skipping clone (running locally)")
    print(f"  Using current directory: {os.getcwd()}")
    print(f"  Make sure you're in the emg2qwerty repository root!")
    
    # Verify we're in a git repo
    if os.path.exists('.git'):
        !git log -1 --oneline
    else:
        print("  ⚠️  Warning: Not a git repository. Make sure you're in the right directory.")

✓ Skipping clone (running locally)
  Using current directory: /Users/sttawm/dev/emg2qwerty
  Make sure you're in the emg2qwerty repository root!
f332469 (HEAD -> main, origin/main, origin/HEAD) Update package versions and kenlm source


---
## 3. Install Dependencies

Install all required Python packages.

**Note:** You may see some dependency warnings in Colab - these can be safely ignored.

In [4]:
# Install requirements
print("Installing dependencies from requirements.txt...")

!pip install -r requirements.txt

print("\n✓ Dependencies installed")

Installing dependencies from requirements.txt...
  Using cached https://github.com/kpu/kenlm/archive/master.zip
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done

[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: pip install --upgrade pip

✓ Dependencies installed


In [5]:
# Verify installation and detect accelerator
import torch
import pytorch_lightning as pl
from hydra import compose, initialize
import h5py
import os

print("✓ Key packages imported successfully")
print(f"  PyTorch version: {torch.__version__}")
print(f"  PyTorch Lightning version: {pl.__version__}")

# Detect available accelerator
if torch.cuda.is_available():
    accelerator = "gpu"
    device_name = torch.cuda.get_device_name(0)
    print(f"  Accelerator: CUDA GPU")
    print(f"  Device: {device_name}")
elif torch.backends.mps.is_available():
    accelerator = "mps"
    # Enable CPU fallback for unsupported ops (like CTC loss)
    os.environ['PYTORCH_ENABLE_MPS_FALLBACK'] = '1'
    print(f"  Accelerator: MPS (Apple Silicon)")
    print(f"  Device: Apple M-series GPU")
    print(f"  Note: CTC loss will fall back to CPU (most ops still run on GPU)")
else:
    accelerator = "cpu"
    print(f"  Accelerator: CPU only")
    print(f"  ⚠️  Training will be slow without GPU/MPS")

# Store accelerator for use in training
ACCELERATOR = accelerator

✓ Key packages imported successfully
  PyTorch version: 2.3.0
  PyTorch Lightning version: 1.8.6
  Accelerator: MPS (Apple Silicon)
  Device: Apple M-series GPU
  Note: CTC loss will fall back to CPU (most ops still run on GPU)


---
## 4. Dataset Setup

Ensure the dataset is available in the `./data` folder.

In [6]:
import os
import shutil

# Change to working directory
if IN_COLAB:
    %cd {WORK_DIR}

if DATASET_PATH and os.path.exists(DATASET_PATH):
    # Option A: Use dataset from specified path (Google Drive or local)
    print(f"Using dataset from: {DATASET_PATH}")
    
    # Create symlink if data folder doesn't exist
    if os.path.exists('./data'):
        print("Removing existing data folder...")
        !rm -rf ./data
    
    print("Creating symlink to dataset...")
    os.symlink(DATASET_PATH, './data')
    print("✓ Dataset linked successfully")
    
elif os.path.exists('./data'):
    print("✓ Dataset folder already exists at ./data")
    
else:
    # Dataset not found - provide manual download instructions
    print("⚠️  Dataset not found at ./data")
    print("\nPlease download the dataset manually:")
    print(f"1. Open this URL in a browser: {DATASET_URL}")
    print("2. Click 'Download' to download the dataset")
    
    if IN_COLAB:
        print("3. Upload to Google Drive (recommended) or Colab session")
        print("4. If using Google Drive:")
        print("   - Set DATASET_PATH in the configuration cell above")
        print("   - Re-run this cell")
        print("   OR extract the dataset to /content/emg2qwerty/data/")
    else:
        print("3. Extract the downloaded archive")
        print("4. Move/copy the extracted 'data' folder to:")
        print(f"   {os.getcwd()}/data/")
        print("   OR set DATASET_PATH in the configuration cell to point to it")
    
    print("\n⚠️  Training will fail without the dataset!")

# Show data directory structure if it exists
if os.path.exists('./data'):
    print("\n✓ Dataset verified!")
    print("\nData directory structure:")
    !ls -lh ./data | head -20

✓ Dataset folder already exists at ./data

✓ Dataset verified!

Data directory structure:
total 9229240
-rw-rw-r--@ 1 sttawm  staff   309M Feb  3  2025 2021-06-02-1622679967-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f.hdf5
-rw-rw-r--@ 1 sttawm  staff   283M Feb  3  2025 2021-06-02-1622681518-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f.hdf5
-rw-rw-r--@ 1 sttawm  staff   293M Feb  3  2025 2021-06-02-1622682789-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f.hdf5
-rw-rw-r--@ 1 sttawm  staff   243M Feb  3  2025 2021-06-03-1622764398-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f.hdf5
-rw-rw-r--@ 1 sttawm  staff   250M Feb  3  2025 2021-06-03-1622765527-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f.hdf5
-rw-rw-r--@ 1 sttawm  staff   243M Feb  3  2025 2021-06-03-1622766673-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f.hdf5
-rw-rw-r--@ 1 sttawm  staff   234M Feb  3  2025 2021-06-04-1622861066-keystrokes-dca-s

---
## 5. Configure Shared Logging

Create the shared logs directory in Google Drive where training logs and checkpoints will be saved.

In [7]:
import os
from datetime import datetime

# Create shared logs directory if it doesn't exist
os.makedirs(SHARED_LOGS_PATH, exist_ok=True)
print(f"✓ Shared logs directory ready: {SHARED_LOGS_PATH}")

# Generate timestamp for this run
timestamp = datetime.now().strftime("%Y-%m-%d/%H-%M-%S")
run_log_dir = os.path.join(SHARED_LOGS_PATH, timestamp)

print(f"\nThis training run will save to:")
print(f"  {run_log_dir}")
print(f"\nCheckpoints will be saved to:")
print(f"  {run_log_dir}/checkpoints/")
print(f"\nAll team members with access to the Google Drive folder can view these logs.")

✓ Shared logs directory ready: ./logs

This training run will save to:
  ./logs/2026-03-04/11-38-31

Checkpoints will be saved to:
  ./logs/2026-03-04/11-38-31/checkpoints/

All team members with access to the Google Drive folder can view these logs.


---
## 6. Training & Monitoring

**Step 1**: Run the TensorBoard cell below to start live monitoring  
**Step 2**: Run the training cell - TensorBoard will update in real-time

**Training details:**
- Model: TDS Convolutional Encoder with CTC loss
- User: Single user (89335547)
- Epochs: 150 (default)
- Batch size: 32
- Accelerator: Automatically detected (CUDA GPU / MPS / CPU)

**Expected training time:**
- CUDA GPU: ~2-4 hours
- Apple M1/M2/M3 (MPS): ~4-8 hours
- CPU: ~24+ hours (not recommended)

In [8]:
# Start TensorBoard for live monitoring (run this BEFORE training)
%load_ext tensorboard

print("Starting TensorBoard...")
print(f"Monitoring logs in: {SHARED_LOGS_PATH}")
print("\nTensorBoard will update live as training progresses.")

%tensorboard --logdir {SHARED_LOGS_PATH}

Starting TensorBoard...
Monitoring logs in: ./logs

TensorBoard will update live as training progresses.


In [ ]:
# Change to working directory
if IN_COLAB:
    %cd {WORK_DIR}

# Construct the training command with logs path
from datetime import datetime
timestamp = datetime.now().strftime("%Y-%m-%d/%H-%M-%S")
log_dir = f"{SHARED_LOGS_PATH}/{timestamp}"

print("Starting training...")
print(f"Environment: {'Google Colab' if IN_COLAB else 'Local'}")
print(f"Accelerator: {ACCELERATOR}")
print(f"Logs and checkpoints will be saved to: {log_dir}")
print("\n" + "="*80 + "\n")

!python -m emg2qwerty.train \
  user="single_user" \
  trainer.accelerator={ACCELERATOR} \
  trainer.devices=1 \
  trainer.max_epochs=150 \
  hydra.run.dir="{log_dir}"

print("\n" + "="*80)
print("\n✓ Training complete!")
print(f"\nCheckpoints saved to: {log_dir}/checkpoints/")
print(f"\nTo access logs, open: {log_dir}")

Starting training...
Environment: Local
Accelerator: mps
Logs and checkpoints will be saved to: ./logs/2026-03-04/11-38-52


[2026-03-04 11:38:54,156][__main__][INFO] - 
Config:
user: single_user
dataset:
  train:
  - user: 89335547
    session: 2021-06-03-1622765527-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f
  - user: 89335547
    session: 2021-06-02-1622681518-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f
  - user: 89335547
    session: 2021-06-04-1622863166-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f
  - user: 89335547
    session: 2021-07-22-1627003020-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f
  - user: 89335547
    session: 2021-07-21-1626916256-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f
  - user: 89335547
    session: 2021-07-22-1627004019-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f
  - user: 89335547
    session: 2021-06-05-1622885888-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4

---
## 7. Post-Training Analysis (Optional)

View checkpoints and metrics after training completes.

In [ ]:
# List saved checkpoints
import glob
from datetime import datetime

timestamp = datetime.now().strftime("%Y-%m-%d/%H-%M-%S")
checkpoint_dir = f"{SHARED_LOGS_PATH}/{timestamp}/checkpoints"

print(f"Checkpoints in {checkpoint_dir}:\n")
if os.path.exists(checkpoint_dir):
    checkpoints = glob.glob(f"{checkpoint_dir}/*.ckpt")
    if checkpoints:
        for ckpt in sorted(checkpoints):
            size = os.path.getsize(ckpt) / (1024 * 1024)  # MB
            print(f"  {os.path.basename(ckpt)} ({size:.1f} MB)")
    else:
        print("  No checkpoints found yet")
else:
    print("  Checkpoint directory not found yet")

---
## 8. Testing (Optional)

Test the trained model with greedy decoding.

In [ ]:
# View latest metrics from PyTorch Lightning logs
import glob
import pandas as pd
from datetime import datetime

timestamp = datetime.now().strftime("%Y-%m-%d/%H-%M-%S")
log_dir = f"{SHARED_LOGS_PATH}/{timestamp}"

# Find tensorboard event files
version_dirs = glob.glob(f"{log_dir}/lightning_logs/version_*")
if version_dirs:
    latest_version = sorted(version_dirs)[-1]
    metrics_file = f"{latest_version}/metrics.csv"
    
    if os.path.exists(metrics_file):
        df = pd.read_csv(metrics_file)
        print("Training metrics (last 10 rows):\n")
        print(df.tail(10).to_string(index=False))
    else:
        print("Metrics file not found yet")
else:
    print("No training logs found yet")

In [ ]:
# Change to working directory
if IN_COLAB:
    %cd {WORK_DIR}

# Specify the checkpoint path
# Replace with the actual path to your best checkpoint
from datetime import datetime
timestamp = datetime.now().strftime("%Y-%m-%d/%H-%M-%S")
CHECKPOINT_PATH = f"{SHARED_LOGS_PATH}/{timestamp}/checkpoints/last.ckpt"

# Or manually specify:
# For Colab: CHECKPOINT_PATH = "/content/drive/MyDrive/emg2qwerty-logs/2024-03-15/10-30-00/checkpoints/epoch=149-step=5000.ckpt"
# For Local: CHECKPOINT_PATH = "./logs/2024-03-15/10-30-00/checkpoints/epoch=149-step=5000.ckpt"

print(f"Testing model from: {CHECKPOINT_PATH}\n")
print("="*80 + "\n")

!python -m emg2qwerty.train \
  user="single_user" \
  checkpoint="{CHECKPOINT_PATH}" \
  train=False \
  trainer.accelerator={ACCELERATOR} \
  decoder=ctc_greedy

print("\n" + "="*80)
print("\n✓ Testing complete!")

---
## Summary

This notebook provides a complete pipeline for:
1. Setting up the EMG2QWERTY training environment (Colab or Local)
2. Training single-user models
3. Saving logs and checkpoints to accessible locations
4. Testing trained models

**Environment-specific behavior:**
- **Google Colab**: Saves logs to Google Drive for team sharing, clones repo fresh each time
- **Local**: Saves logs to `./logs/` directory, uses your existing repo

**Next steps:**
- **Colab users**: Share the Google Drive logs folder with your team
- **Local users**: Logs are in `./logs/` directory in your repo
- Compare results across different training runs
- Experiment with different hyperparameters by modifying the training command

**For more information:**
- Repository: https://github.com/sttawm/emg2qwerty
- Paper: [EMG-to-Speech: Direct Generation of Speech From Facial Electromyographic Signals](https://arxiv.org/abs/2108.06626)